# Sesión 3B · Agrupar y cruzar bases

Ya tenemos el dengue limpio. Ahora la pregunta completa:

> **¿Qué distritos tienen muchos casos de dengue y pocos establecimientos de salud?**

Para responderla hay que **agrupar** (pasar de 172 mil filas a una por distrito)
y **cruzar** con la base de establecimientos del MINSA.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

dengue = pd.read_csv("dengue_limpio.csv", dtype={"ubigeo": str})
dengue.head(3)

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos,brote
0,2020,43,Dengue,Cusco,MEGANTONI,LA CONVENCION,080914,2.0,False
1,2021,41,Dengue,Moquegua,MOQUEGUA,MARISCAL NIETO,180101,0.0,False
2,2021,42,Dengue,Moquegua,MOQUEGUA,MARISCAL NIETO,180101,0.0,False


## 1. `groupby`: de muchas filas a un resumen

La idea es siempre la misma: **agrupar por** algo, **calcular** algo.

In [2]:
dengue.groupby("departamento")["casos"].sum().head(10)

departamento
Amazonas        3217.0
Ancash          2344.0
Ayacucho        8818.0
Cajamarca       4144.0
Callao            11.0
Cusco           5916.0
Huanuco         5695.0
Ica            16588.0
Junin           9841.0
La Libertad    13844.0
Name: casos, dtype: float64

### Ordenado, que es como se lee de verdad

In [3]:
dengue.groupby("departamento")["casos"].sum().sort_values(ascending=False).head(10)

departamento
Piura            66309.0
Loreto           23570.0
Tumbes           17692.0
Ica              16588.0
Ucayali          15214.0
Madre De Dios    14878.0
La Libertad      13844.0
San Martin       13677.0
Junin             9841.0
Ayacucho          8818.0
Name: casos, dtype: float64

### Varias operaciones a la vez con `agg`

In [4]:
dengue.groupby("departamento")["casos"].agg(["sum", "mean", "max"]).sort_values(
    "sum", ascending=False
).head(10)

,sum,mean,max
departamento,,,
Piura,66309.0,2.889407,912.0
Loreto,23570.0,1.323562,250.0
Tumbes,17692.0,3.668256,206.0
Ica,16588.0,1.208421,444.0
Ucayali,15214.0,2.496144,266.0
Madre De Dios,14878.0,4.010243,317.0
La Libertad,13844.0,1.332692,263.0
San Martin,13677.0,0.491536,129.0
Junin,9841.0,2.159061,123.0


### Agrupar por dos columnas

El resultado tiene un índice de dos niveles. `reset_index()` lo vuelve tabla normal.

In [5]:
por_depto_anio = (
    dengue.groupby(["departamento", "anio"])["casos"].sum().reset_index()
)
por_depto_anio.head()

,departamento,anio,casos
0,Amazonas,2015,37.0
1,Amazonas,2016,90.0
2,Amazonas,2017,93.0
3,Amazonas,2018,109.0
4,Amazonas,2019,164.0


## 2. `pivot_table`: el cuadro cruzado de toda la vida

Lo mismo de arriba, pero con los años como columnas. Esto es lo que se pega
en un informe.

In [6]:
cuadro = dengue.pivot_table(
    index="departamento", columns="anio", values="casos", aggfunc="sum", fill_value=0
)
cuadro.head(10)

anio,2015,2016,2017,2018,2019,2020,2021
departamento,,,,,,,
Amazonas,37.0,90.0,93.0,109.0,164.0,894.0,1830.0
Ancash,118.0,454.0,1720.0,6.0,20.0,0.0,26.0
Ayacucho,268.0,2638.0,1657.0,202.0,95.0,2612.0,1346.0
Cajamarca,218.0,281.0,420.0,6.0,399.0,265.0,2555.0
Callao,0.0,0.0,5.0,0.0,0.0,4.0,2.0
Cusco,248.0,1100.0,537.0,79.0,54.0,2557.0,1341.0
Huanuco,307.0,728.0,92.0,25.0,35.0,1773.0,2735.0
Ica,3.0,323.0,4384.0,127.0,51.0,7123.0,4577.0
Junin,774.0,931.0,220.0,51.0,481.0,4216.0,3168.0


### Con totales

In [7]:
cuadro_total = dengue.pivot_table(
    index="departamento",
    columns="anio",
    values="casos",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)
cuadro_total.sort_values("Total", ascending=False).head(10)

anio,2015,2016,2017,2018,2019,2020,2021,Total
departamento,,,,,,,,
Total,35817.0,25160.0,57858.0,4698.0,15290.0,56103.0,36196.0,231122.0
Piura,20043.0,7610.0,33843.0,525.0,70.0,208.0,4010.0,66309.0
Loreto,1630.0,1686.0,1089.0,1833.0,2548.0,10678.0,4106.0,23570.0
Tumbes,7418.0,1089.0,4145.0,64.0,508.0,3104.0,1364.0,17692.0
Ica,3.0,323.0,4384.0,127.0,51.0,7123.0,4577.0,16588.0
Ucayali,350.0,1007.0,779.0,317.0,214.0,10934.0,1613.0,15214.0
Madre De Dios,966.0,468.0,565.0,1234.0,7399.0,3363.0,883.0,14878.0
La Libertad,2073.0,4650.0,5904.0,3.0,366.0,466.0,382.0,13844.0
San Martin,220.0,335.0,460.0,98.0,1969.0,6629.0,3966.0,13677.0


## 3. Resumir por distrito

Este es el nivel al que vamos a cruzar. Una fila por distrito.

In [8]:
por_distrito = (
    dengue.groupby(["ubigeo", "departamento", "provincia", "distrito"])["casos"]
    .sum()
    .reset_index()
    .rename(columns={"casos": "casos_dengue"})
)
por_distrito.shape

(475, 5)

In [9]:
por_distrito.sort_values("casos_dengue", ascending=False).head(10)

,ubigeo,departamento,provincia,distrito,casos_dengue
348,200601,Piura,SULLANA,SULLANA,10989.0
307,200104,Piura,PIURA,CASTILLA,9327.0
306,200101,Piura,PIURA,PIURA,8847.0
289,170101,Madre De Dios,TAMBOPATA,TAMBOPATA,8684.0
445,240101,Tumbes,TUMBES,TUMBES,7684.0
458,250101,Ucayali,CORONEL PORTILLO,CALLERIA,6212.0
252,160201,Loreto,ALTO AMAZONAS,YURIMAGUAS,5474.0
315,200115,Piura,PIURA,VEINTISEIS DE OCTUBRE,5194.0
331,200401,Piura,MORROPON,CHULUCANAS,4552.0
314,200114,Piura,PIURA,TAMBO GRANDE,4395.0


## 4. La segunda base: establecimientos de salud

Registro nacional del MINSA (RENAES).

In [10]:
salud = pd.read_csv("../../data/03-pandas/establecimientos_salud.csv", encoding="utf-8-sig")
salud.head(3)

,Institución,Nombre del establecimiento,Clasificación,Departamento,Provincia,Distrito,UBIGEO,Categoria,CAMAS,Estado
0,GOBIERNO REGIONAL,SAN PABLO,PUESTOS DE SALUD O POSTAS DE SALUD,PIURA,TALARA,LA BREA,200703,I-1,NaN,ACTIVADO
1,ESSALUD,POSTA MEDICA NEGRITOS,PUESTOS DE SALUD O POSTAS DE SALUD,PIURA,TALARA,LA BREA,200703,Sin Categoría,NaN,ACTIVADO
2,GOBIERNO REGIONAL,NEGRITOS,CENTROS DE SALUD O CENTROS MEDICOS,PIURA,TALARA,LA BREA,200703,I-3,NaN,ACTIVADO


In [11]:
salud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8464 entries, 0 to 8463
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Institución                 8464 non-null   object 
 1   Nombre del establecimiento  8464 non-null   object 
 2   Clasificación               8460 non-null   object 
 3   Departamento                8464 non-null   object 
 4   Provincia                   8464 non-null   object 
 5   Distrito                    8464 non-null   object 
 6   UBIGEO                      8464 non-null   int64  
 7   Categoria                   8464 non-null   object 
 8   CAMAS                       537 non-null    float64
 9   Estado                      8464 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 661.4+ KB


### Mismo problema del ubigeo: hay que dejarlo igual en ambas bases

Si en una base el ubigeo es `080914` y en la otra `80914`, el cruce falla en
silencio y uno se entera tarde.

In [12]:
salud["UBIGEO"] = salud["UBIGEO"].astype(str).str.zfill(6)
salud["UBIGEO"].head()

0    200703
1    200703
2    200703
3    200704
4    200701
Name: UBIGEO, dtype: object

### ¿Qué tipos de establecimiento hay?

In [13]:
salud["Clasificación"].value_counts().head()

Clasificación
PUESTOS DE SALUD O POSTAS DE SALUD                 6493
CENTROS DE SALUD O CENTROS MEDICOS                 1251
CENTROS DE SALUD CON CAMAS DE INTERNAMIENTO         324
HOSPITALES O CLINICAS DE ATENCION GENERAL           197
HOSPITALES O CLINICAS DE ATENCION ESPECIALIZADA      54
Name: count, dtype: int64

In [14]:
salud["Categoria"].value_counts().head(10)

Categoria
I-1              4350
I-2              2101
I-3              1298
I-4               325
Sin Categoría     146
II-1              137
II-2               50
III-1              27
III-2              12
II-E               12
Name: count, dtype: int64

### Resumir a una fila por distrito

In [15]:
salud_distrito = (
    salud.groupby("UBIGEO")
    .agg(n_establecimientos=("Nombre del establecimiento", "count"),
         camas=("CAMAS", "sum"))
    .reset_index()
    .rename(columns={"UBIGEO": "ubigeo"})
)
salud_distrito.head()

,ubigeo,n_establecimientos,camas
0,010101,12,140.0
1,010102,1,0.0
2,010103,3,0.0
3,010104,1,0.0
4,010105,4,0.0


## 5. `merge`: cruzar las dos bases

`on` es la columna común. `how` decide qué filas se conservan:

| `how` | Se queda con |
|---|---|
| `"inner"` | solo lo que está en ambas |
| `"left"` | todo lo de la izquierda |
| `"outer"` | todo de ambas |

In [16]:
tabla = por_distrito.merge(salud_distrito, on="ubigeo", how="left")
tabla.head()

,ubigeo,departamento,provincia,distrito,casos_dengue,n_establecimientos,camas
0,010101,Amazonas,CHACHAPOYAS,CHACHAPOYAS,6.0,12.0,140.0
1,010201,Amazonas,BAGUA,BAGUA,949.0,11.0,103.0
2,010202,Amazonas,BAGUA,ARAMANGO,49.0,19.0,0.0
3,010203,Amazonas,BAGUA,COPALLIN,47.0,6.0,0.0
4,010204,Amazonas,BAGUA,EL PARCO,10.0,3.0,0.0


### Siempre revisar qué no cruzó

Este es el paso que la gente se salta y luego reporta números mal.

In [17]:
tabla["n_establecimientos"].isna().sum()

np.int64(3)

In [18]:
tabla[tabla["n_establecimientos"].isna()].head()

,ubigeo,departamento,provincia,distrito,casos_dengue,n_establecimientos,camas
81,080912,Cusco,LA CONVENCION,VILLA VIRGEN,33.0,NaN,NaN
82,080913,Cusco,LA CONVENCION,VILLA KINTIARINA,6.0,NaN,NaN
158,120609,Junin,SATIPO,VIZCATAN DEL ENE,26.0,NaN,NaN


Los distritos sin establecimientos registrados quedan en cero.

In [19]:
tabla["n_establecimientos"] = tabla["n_establecimientos"].fillna(0)
tabla["camas"] = tabla["camas"].fillna(0)
tabla.isna().sum()

ubigeo                0
departamento          0
provincia             0
distrito              0
casos_dengue          0
n_establecimientos    0
camas                 0
dtype: int64

## 6. La respuesta a la pregunta

Casos de dengue por cada establecimiento de salud del distrito.

In [20]:
tabla["casos_por_establecimiento"] = (
    tabla["casos_dengue"] / tabla["n_establecimientos"].replace(0, pd.NA)
)
tabla.head()

,ubigeo,departamento,provincia,distrito,casos_dengue,n_establecimientos,camas,casos_por_establecimiento
0,010101,Amazonas,CHACHAPOYAS,CHACHAPOYAS,6.0,12.0,140.0,0.5
1,010201,Amazonas,BAGUA,BAGUA,949.0,11.0,103.0,86.272727
2,010202,Amazonas,BAGUA,ARAMANGO,49.0,19.0,0.0,2.578947
3,010203,Amazonas,BAGUA,COPALLIN,47.0,6.0,0.0,7.833333
4,010204,Amazonas,BAGUA,EL PARCO,10.0,3.0,0.0,3.333333


### Los 15 distritos con más presión sobre su sistema de salud

Se filtran los que tienen al menos 100 casos, para no premiar distritos
minúsculos con un solo caso.

In [21]:
criticos = (
    tabla[tabla["casos_dengue"] >= 100]
    .sort_values("casos_por_establecimiento", ascending=False)
    .head(15)
)
criticos[["departamento", "provincia", "distrito", "casos_dengue",
          "n_establecimientos", "casos_por_establecimiento"]]

,departamento,provincia,distrito,casos_dengue,n_establecimientos,casos_por_establecimiento
202,Lambayeque,CHICLAYO,TUMAN,1464.0,1.0,1464.0
306,Piura,PIURA,PIURA,8847.0,7.0,1263.857143
92,Huanuco,LEONCIO PRADO,RUPA-RUPA,3296.0,3.0,1098.666667
356,Piura,TALARA,PARIÑAS,3997.0,4.0,999.25
445,Tumbes,TUMBES,TUMBES,7684.0,8.0,960.5
348,Piura,SULLANA,SULLANA,10989.0,12.0,915.75
361,Piura,TALARA,MANCORA,785.0,1.0,785.0
454,Tumbes,ZARUMILLA,ZARUMILLA,1428.0,2.0,714.0
435,San Martin,SAN MARTIN,MORALES,1391.0,2.0,695.5
174,La Libertad,CHEPEN,CHEPEN,2054.0,3.0,684.666667


## 7. `concat`: apilar tablas

`merge` pega columnas; `concat` pega filas. Sirve cuando se tiene un archivo
por año y hay que juntarlos.

In [22]:
a = dengue[dengue["anio"] == 2020]
b = dengue[dengue["anio"] == 2021]
juntos = pd.concat([a, b])
print(a.shape, b.shape, juntos.shape)

(24751, 9) (24857, 9) (49608, 9)


## 8. Exportar a Excel

Varias hojas en un solo archivo: así se entrega un informe.

In [23]:
with pd.ExcelWriter("reporte_dengue.xlsx") as writer:
    cuadro_total.to_excel(writer, sheet_name="Por departamento")
    criticos.to_excel(writer, sheet_name="Distritos criticos", index=False)
    tabla.to_excel(writer, sheet_name="Base completa", index=False)

tabla.to_csv("tabla_distritos.csv", index=False)
print("Listo:", tabla.shape)

Listo: (475, 8)


---
## Lo que hicimos

| Paso | Código |
|---|---|
| Agrupar | `.groupby("col")["valor"].sum()` |
| Varias métricas | `.agg(["sum", "mean", "max"])` |
| Nombrar columnas | `.agg(nueva=("col", "sum"))` |
| Cuadro cruzado | `.pivot_table(index=, columns=, values=, aggfunc=)` |
| Cruzar bases | `.merge(otra, on="llave", how="left")` |
| Revisar el cruce | `.isna().sum()` **después** del merge |
| Apilar | `pd.concat([a, b])` |
| Exportar | `pd.ExcelWriter` |

### La regla que más importa

Después de todo `merge`, contar cuántas filas no cruzaron. Un cruce que falla
no da error: da números equivocados.

---
## Tarea 1

Ya puedes hacerla: está en `assignments/tarea-1-pandas.md`.